In [ ]:
import os
import re
import json
from datasets import load_dataset

In [ ]:
dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

In [ ]:
def extract_modified_file_path(patch):
    """Extracts the modified file path from the first line of a Git diff."""
    match = re.search(r'diff --git a/(.*?) b/', patch)
    return match.group(1) if match else None

In [ ]:
bug_paths = {

}

for index, task in enumerate(dataset):
    patch = task["patch"]
    instance_id = task["instance_id"]
    file_path = extract_modified_file_path(patch)

    bug_paths[instance_id] = file_path

In [ ]:

def extract_module_description(file_path):
    """Extract the module-level docstring from a Python file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        # Look for triple-quoted docstrings at the module level
        # This pattern matches both """ and ''' style docstrings
        docstring_pattern = re.compile(r'^(?:(?:#[^\n]*\n)*)?(?:\"\"\"|\'\'\')(.*?)(?:\"\"\"|\'\'\')(?:\s*$|\s*\n)', 
                                      re.DOTALL | re.MULTILINE)
        
        match = docstring_pattern.search(content)
        if match:
            # Extract the docstring and clean it up
            docstring = match.group(1).strip()
            
            # Convert multiline docstring to a single line
            docstring = '   '.join([line.strip() for line in docstring.split('\n')])
            
            # Truncate if too long (optional)
            # if len(docstring) > 120:
            #     docstring = docstring[:117] + "..."
                
            return docstring
    except Exception:
        pass
    
    return None

def get_folder_description(path, show_descriptions=False):
    """Get the description for a folder from its __init__.py file."""
    if not show_descriptions:
        return None
        
    init_path = os.path.join(path, "__init__.py")
    if os.path.isfile(init_path):
        return extract_module_description(init_path)
    
    return None

def get_tree(path, indent="", is_last=True, file_extensions=None, ignored_dirs=None, show_descriptions=False):
    name = os.path.basename(path)
    
    # Skip ignored directories
    if ignored_dirs and os.path.isdir(path) and name.lower() in ignored_dirs:
        return []
    
    # Check if it's a file or directory
    if os.path.isdir(path):
        # Get all items in the directory
        try:
            items = os.listdir(path)
        except PermissionError:
            return []
            
        items.sort()
        
        # Check if this directory contains any Python files
        contains_python = False
        for item in items:
            item_path = os.path.join(path, item)
            if os.path.isfile(item_path) and os.path.splitext(item)[1].lower() in file_extensions:
                contains_python = True
                break
            elif os.path.isdir(item_path) and not (ignored_dirs and item.lower() in ignored_dirs):
                # Check subdirectories recursively
                subtree = get_tree(item_path, "", True, file_extensions, ignored_dirs, show_descriptions)
                if subtree:
                    contains_python = True
                    break
        
        # If no Python files in this directory or subdirectories, skip it
        if not contains_python:
            return []
        
        # Create the line for the current directory
        line = indent
        line += "└── " if is_last else "├── "
        line += name
        
        # Add description if available
        description = get_folder_description(path, show_descriptions)
        if description:
            line += f"  # {description}"
        
        result = [line]
        
        # Process each item
        valid_items = []
        for item in items:
            item_path = os.path.join(path, item)
            
            # Skip hidden files starting with "."
            if item.startswith('.'):
                continue
                
            # Skip ignored directories
            if ignored_dirs and os.path.isdir(item_path) and item.lower() in ignored_dirs:
                continue
                
            # Only include python files
            if os.path.isfile(item_path):
                ext = os.path.splitext(item)[1].lower()
                if ext in file_extensions:
                    valid_items.append(item)
            else:
                # Include directories that might contain python files
                subtree = get_tree(item_path, "", True, file_extensions, ignored_dirs, show_descriptions)
                if subtree:
                    valid_items.append(item)
        
        # Process valid items
        for i, item in enumerate(valid_items):
            item_path = os.path.join(path, item)
            
            # Determine if this is the last item
            is_last_item = (i == len(valid_items) - 1)
            
            # Create new indent for the next level
            new_indent = indent + ("    " if is_last else "│   ")
            
            # Add the item to the result
            subtree = get_tree(item_path, new_indent, is_last_item, file_extensions, ignored_dirs, show_descriptions)
            result.extend(subtree)
        
        return result
    else:
        # If it's a file, just return the line
        ext = os.path.splitext(name)[1].lower()
        if ext in file_extensions:
            line = indent
            line += "└── " if is_last else "├── "
            line += name
            return [line]
        else:
            return []

def build_tree_dict(path, file_extensions=None, ignored_dirs=None, show_descriptions=False):
    name = os.path.basename(path)
    
    # Skip ignored directories
    if ignored_dirs and os.path.isdir(path) and name.lower() in ignored_dirs:
        return None
    
    if os.path.isdir(path):
        # Get all items in the directory
        try:
            items = os.listdir(path)
        except PermissionError:
            return None
            
        items.sort()
        
        # Check if this directory contains any Python files
        contains_python = False
        for item in items:
            item_path = os.path.join(path, item)
            if os.path.isfile(item_path) and os.path.splitext(item)[1].lower() in file_extensions:
                contains_python = True
                break
            elif os.path.isdir(item_path) and not (ignored_dirs and item.lower() in ignored_dirs):
                # Check subdirectories recursively
                child = build_tree_dict(item_path, file_extensions, ignored_dirs, show_descriptions)
                if child:
                    contains_python = True
                    break
        
        # If no Python files in this directory or subdirectories, skip it
        if not contains_python:
            return None
        
        result = {"name": name, "type": "directory", "children": []}
        
        # Add description if available
        if show_descriptions:
            description = get_folder_description(path, show_descriptions)
            result["description"] = description if description else ""
        
        for item in items:
            item_path = os.path.join(path, item)
            
            # Skip hidden files
            if item.startswith('.'):
                continue
                
            # Skip ignored directories
            if ignored_dirs and os.path.isdir(item_path) and item.lower() in ignored_dirs:
                continue
                
            # Process files and directories
            child = build_tree_dict(item_path, file_extensions, ignored_dirs, show_descriptions)
            if child:
                result["children"].append(child)
        
        return result
    else:
        # If it's a file, check if it's a Python file
        ext = os.path.splitext(name)[1].lower()
        if ext in file_extensions:
            return {"name": name, "type": "file"}
        else:
            return None

def generate_trees(path, show_descriptions=False):
    # Define ignored directories and file extensions
    ignored_dirs = ["config", "test", "tests", "example", "examples"]
    py_extensions = ['.py']
    
    # 1. Python Files Tree
    print("Python Files Tree: ")
    tree_py = get_tree(path, file_extensions=py_extensions, ignored_dirs=ignored_dirs, 
                      show_descriptions=show_descriptions)

    if tree_py:
        for line in tree_py:
            print(line)
    else:
        print("No Python files found.")
    print("\n")
    
    # 2. Python Files as JSON
    print("Python Files as JSON:")
    tree_dict_py = build_tree_dict(path, file_extensions=py_extensions, ignored_dirs=ignored_dirs,
                                 show_descriptions=show_descriptions)

    if tree_dict_py:
        print(json.dumps(tree_dict_py, indent=2))
    else:
        print("{}")

    
path = './codebases/astropy__astropy-6938'
show_descriptions = True

generate_trees(path, show_descriptions)